In [6]:
import cobra
import pandas as pd

def loadMetaboliteAnnotations(modelPath):
    model = cobra.io.read_sbml_model(modelPath)
    records = []
    for met in model.metabolites:
        records.append({
            "metID"      : met.id,
            "name"       : met.name,
            "formula"    : met.formula,
            "compartment": met.compartment,
            "inchi"      : met.annotation.get("inchi"),
            "kegg"       : met.annotation.get("kegg.compound"),
            "chebi"      : met.annotation.get("chebi"),
            "biggID"     : met.annotation.get("bigg.metabolite"),
        })
    return pd.DataFrame(records)

Ecoli_metabolites  = loadMetaboliteAnnotations("iAF1260.xml")
Putida_metabolites = loadMetaboliteAnnotations("iJN1463.xml")
#Human_metabolites  = loadMetaboliteAnnotations("Recon3D.xml")   

Ecoli_metabolites["source"]  = "Ecoli"
Putida_metabolites["source"] = "Putida"
#Human_metabolites["source"]  = "Human"

allMetabolitesDF = pd.concat(
    [Ecoli_metabolites, Putida_metabolites],
    ignore_index=True,
)
allMetabolitesDF

,metID,name,formula,compartment,inchi,kegg,chebi,biggID,source
0,2agpg161_c,2-Acyl-sn-glycero-3-phosphoglycerol (n-C16:1),C22H42O9P1,c,None,None,None,2agpg161,Ecoli
1,2agpg161_p,2-Acyl-sn-glycero-3-phosphoglycerol (n-C16:1),C22H42O9P1,p,None,None,None,2agpg161,Ecoli
2,2agpg180_c,2-Acyl-sn-glycero-3-phosphoglycerol (n-C18:0),C24H48O9P1,c,None,None,None,2agpg180,Ecoli
3,2agpg180_p,2-Acyl-sn-glycero-3-phosphoglycerol (n-C18:0),C24H48O9P1,p,None,None,None,2agpg180,Ecoli
4,2agpg181_c,2-Acyl-sn-glycero-3-phosphoglycerol (n-C18:1),C24H46O9P1,c,None,None,None,2agpg181,Ecoli
...,...,...,...,...,...,...,...,...,...
3816,mtsoxin_c,Methionine sulfoximine,C5H12N2O3S,c,None,None,None,mtsoxin,Putida
3817,mtsoxin_p,Methionine sulfoximine,C5H12N2O3S,p,None,None,None,mtsoxin,Putida
3818,mtsoxin_e,Methionine sulfoximine,C5H12N2O3S,e,None,None,None,mtsoxin,Putida
3819,acmtsoxin_c,N Acetylmethionine sulfoximine,C7H13N2O4S,c,None,None,None,acmtsoxin,Putida


In [7]:
def fetchSmilesFromMetaNetX(metabolitesDF, chemXrefPath, chemPropPath):
    chemXrefDF = pd.read_csv(
        chemXrefPath, sep="\t", comment="#",
        names=["source", "mnxID", "description"],
    )
    biggXrefDF = chemXrefDF[chemXrefDF["source"].str.startswith("bigg.metabolite:", na=False)].copy()
    biggXrefDF["biggID"] = biggXrefDF["source"].str.replace("bigg.metabolite:", "", regex=False)
    biggToMnx = biggXrefDF.drop_duplicates("biggID").set_index("biggID")["mnxID"]

    chemPropDF = pd.read_csv(
        chemPropPath, sep="\t", comment="#",
        names=["mnxID", "name", "reference", "formula",
               "charge", "mass", "inchi", "inchikey", "smiles"],
    )
    mnxToSmiles = chemPropDF.drop_duplicates("mnxID").set_index("mnxID")["smiles"]

    result = metabolitesDF.copy()
    result["mnxID"] = result["biggID"].map(biggToMnx)
    result["smiles"] = result["mnxID"].map(mnxToSmiles)
    return result

allMetabolitesDF = fetchSmilesFromMetaNetX(
    allMetabolitesDF,
    chemXrefPath="chem_xref.tsv",
    chemPropPath="chem_prop.tsv",
)
allMetabolitesDF

,metID,name,formula,compartment,inchi,kegg,chebi,biggID,source,mnxID,smiles
0,2agpg161_c,2-Acyl-sn-glycero-3-phosphoglycerol (n-C16:1),C22H42O9P1,c,None,None,None,2agpg161,Ecoli,MNXM3453,NaN
1,2agpg161_p,2-Acyl-sn-glycero-3-phosphoglycerol (n-C16:1),C22H42O9P1,p,None,None,None,2agpg161,Ecoli,MNXM3453,NaN
2,2agpg180_c,2-Acyl-sn-glycero-3-phosphoglycerol (n-C18:0),C24H48O9P1,c,None,None,None,2agpg180,Ecoli,MNXM729358,NaN
3,2agpg180_p,2-Acyl-sn-glycero-3-phosphoglycerol (n-C18:0),C24H48O9P1,p,None,None,None,2agpg180,Ecoli,MNXM729358,NaN
4,2agpg181_c,2-Acyl-sn-glycero-3-phosphoglycerol (n-C18:1),C24H46O9P1,c,None,None,None,2agpg181,Ecoli,MNXM3455,CCCCCCC=CCCCCCCCCCC(=O)OCC(O)COP(=O)([O-])OCC(...
...,...,...,...,...,...,...,...,...,...,...,...
3816,mtsoxin_c,Methionine sulfoximine,C5H12N2O3S,c,None,None,None,mtsoxin,Putida,MNXM1093398,NaN
3817,mtsoxin_p,Methionine sulfoximine,C5H12N2O3S,p,None,None,None,mtsoxin,Putida,MNXM1093398,NaN
3818,mtsoxin_e,Methionine sulfoximine,C5H12N2O3S,e,None,None,None,mtsoxin,Putida,MNXM1093398,NaN
3819,acmtsoxin_c,N Acetylmethionine sulfoximine,C7H13N2O4S,c,None,None,None,acmtsoxin,Putida,MNXM1092516,NaN


In [8]:
from rdkit import Chem

def canonSmiles(smi):
    mol = Chem.MolFromSmiles(str(smi))
    return Chem.MolToSmiles(mol) if mol else str(smi)

allMetabolitesDF = allMetabolitesDF[["biggID", "name", "mnxID", "smiles", "source"]]

def isValidSmiles(smi):
    if pd.isna(smi):
        return False
    return Chem.MolFromSmiles(str(smi)) is not None

allMetabolitesDF = allMetabolitesDF[
    allMetabolitesDF["smiles"].apply(isValidSmiles)
].reset_index(drop=True)

allMetabolitesDF["smiles"] = allMetabolitesDF["smiles"].apply(canonSmiles)
allMetabolitesDF = allMetabolitesDF.rename(columns={"smiles": "SMILES"})

allMetabolitesDF = allMetabolitesDF.drop_duplicates(
    subset="SMILES", keep="first"
).reset_index(drop=True)
allMetabolitesDF

[17:50:46] WARNING: not removing hydrogen atom with dummy atom neighbors
[17:50:46] WARNING: not removing hydrogen atom with dummy atom neighbors
[17:50:46] WARNING: not removing hydrogen atom with dummy atom neighbors
[17:50:46] WARNING: not removing hydrogen atom with dummy atom neighbors
[17:50:46] WARNING: not removing hydrogen atom with dummy atom neighbors
[17:50:46] WARNING: not removing hydrogen atom with dummy atom neighbors
[17:50:46] WARNING: not removing hydrogen atom with dummy atom neighbors
[17:50:46] WARNING: not removing hydrogen atom with dummy atom neighbors
[17:50:46] WARNING: not removing hydrogen atom with dummy atom neighbors
[17:50:46] WARNING: not removing hydrogen atom with dummy atom neighbors
[17:50:46] WARNING: not removing hydrogen atom with dummy atom neighbors
[17:50:46] WARNING: not removing hydrogen atom with dummy atom neighbors
[17:50:46] WARNING: not removing hydrogen atom with dummy atom neighbors
[17:50:46] WARNING: not removing hydrogen atom with

,biggID,name,mnxID,SMILES,source
0,2agpg181,2-Acyl-sn-glycero-3-phosphoglycerol (n-C18:1),MNXM3455,CCCCCCC=CCCCCCCCCCC(=O)OCC(O)COP(=O)([O-])OCC(...,Ecoli
1,2ahbut,(S)-2-Aceto-2-hydroxybutanoate,MNXM726902,CC[C@](O)(C(C)=O)C(=O)[O-],Ecoli
2,2amsa,2-Aminomalonate semialdehyde,MNXM2124,[NH3+][C@@H](C=O)C(=O)[O-],Ecoli
3,2aobut,L-2-Amino-3-oxobutanoate,MNXM114087,CC(=O)[C@H]([NH3+])C(=O)[O-],Ecoli
4,2cpr5p,1-(2-Carboxyphenylamino)-1-deoxy-D-ribulose 5-...,MNXM1455,O=C([O-])c1ccccc1NCC(=O)[C@H](O)[C@H](O)COP(=O...,Ecoli
...,...,...,...,...,...
1158,vacccoa,Vaccenyl coenzyme A,MNXM4868,CCCCCC/C=C\CCCCCCCCCC(=O)SCCN=C(O)CCN=C(O)C(O)...,Putida
1159,vanln,Vanillin,MNXM754,COc1cc(C=O)ccc1O,Putida
1160,vanlt,Vanillate,MNXM982,COc1cc(C(=O)[O-])ccc1O,Putida
1161,S2hglut,(S)-2-Hydroxyglutarate,MNXM733231,O=C([O-])CC[C@H](O)C(=O)[O-],Putida


In [9]:
doranetNativeDF = pd.read_csv("doranet_all_cofactors.tsv", sep="\t")
doranetNativeDF["canonKey"] = doranetNativeDF["SMILES"].apply(canonSmiles)
doranetNativeSet = set(doranetNativeDF["canonKey"])

doranetCofactorsDF = allMetabolitesDF[["biggID", "name", "SMILES"]].copy()
doranetCofactorsDF.columns = ["#ID", "Name", "SMILES"]

overlapMask = doranetCofactorsDF["SMILES"].isin(doranetNativeSet)
print(f"Metabolites also present in DORAnet native cofactors: {overlapMask.sum()}")

doranetCofactorsDF.to_csv("custom_cofactors.tsv", sep="\t", index=False)
print(f"Custom cofactors written: {len(doranetCofactorsDF)} entries")

Metabolites also present in DORAnet native cofactors: 7
Custom cofactors written: 1163 entries


[17:50:47] WARNING: not removing hydrogen atom without neighbors
